# 05 - Visualizacao da Segmentacao Bruta

Le os resumos persistidos pelo notebook 04 e reconstrói a base linha a linha em memoria para gerar os graficos iniciais da segmentacao bruta.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis import MetricsCollector
from src.repositories import (
    AnaliseSegmentacaoBrutaResumoExecucaoRepository,
    AnaliseSegmentacaoBrutaResumoModeloRepository,
    AnaliseSegmentacaoBrutaResumoTagRepository,
)
from src.visualization import (
    plot_metric_bars_by_model,
    plot_metric_by_execution_heatmap,
    plot_metric_distribution_by_model,
    plot_metric_distribution_by_tag,
    plot_metric_tag_comparison,
)


## Carregamento dos resumos e da base linha a linha

Os resumos agregados sao lidos do SQLite. A base linha a linha e reconstruida apenas para os graficos de distribuicao.


In [ ]:
def _registros_para_dataframe(registros, columns):
    return pd.DataFrame([{column: getattr(registro, column) for column in columns} for registro in registros])

metric_names = ['auprc', 'soft_dice', 'brier_score']
tags_prioritarias = [
    'tag_multi_bufalos',
    'tag_baixo_contraste',
    'tag_angulo_extremo',
    'tag_cortado',
    'tag_ocluido',
]

collector = MetricsCollector(force_recalculate=False)
df_base = collector.collect_all_metrics()

resumo_modelo_repository = AnaliseSegmentacaoBrutaResumoModeloRepository()
resumo_execucao_repository = AnaliseSegmentacaoBrutaResumoExecucaoRepository()
resumo_tag_repository = AnaliseSegmentacaoBrutaResumoTagRepository()

df_resumo_modelo = _registros_para_dataframe(
    resumo_modelo_repository.list(),
    [
        'nome_modelo', 'metric_name', 'count', 'mean', 'median', 'std',
        'min', 'max', 'q1', 'q3', 'iqr', 'higher_is_better'
    ],
)
df_resumo_execucao = _registros_para_dataframe(
    resumo_execucao_repository.list(),
    [
        'nome_modelo', 'execucao', 'metric_name', 'count', 'mean', 'median',
        'std', 'min', 'max', 'q1', 'q3', 'iqr', 'higher_is_better'
    ],
)
df_resumo_tag = _registros_para_dataframe(
    resumo_tag_repository.list(),
    [
        'nome_modelo', 'tag_name', 'tag_value', 'metric_name', 'count', 'mean',
        'median', 'std', 'min', 'max', 'q1', 'q3', 'iqr', 'higher_is_better'
    ],
)

print(f'Registros da base linha a linha: {len(df_base)}')
print(f'Resumo por modelo: {len(df_resumo_modelo)}')
print(f'Resumo por execucao: {len(df_resumo_execucao)}')
print(f'Resumo por tag: {len(df_resumo_tag)}')


## Visao geral por modelo

A tabela abaixo resume as medias e medianas por metrica, e os graficos ajudam a comparar os modelos de forma agregada.


In [ ]:
display(
    df_resumo_modelo.pivot(
        index='nome_modelo',
        columns='metric_name',
        values='mean',
    ).sort_index()
)

display(
    df_resumo_modelo.pivot(
        index='nome_modelo',
        columns='metric_name',
        values='median',
    ).sort_index()
)

for metric_name in metric_names:
    fig, _ = plot_metric_bars_by_model(df_resumo_modelo, metric_name)
    plt.show()


## Estabilidade por execucao

Aqui avaliamos se o comportamento do modelo se mantém estável entre as execucoes.


In [ ]:
for metric_name in metric_names:
    fig, _ = plot_metric_by_execution_heatmap(df_resumo_execucao, metric_name)
    plt.show()

display(
    df_resumo_execucao.pivot_table(
        index='nome_modelo',
        columns=['metric_name', 'execucao'],
        values='mean',
    ).sort_index()
)


## Impacto das tags

As tags entram como eixo explicativo do que degrada a segmentacao. O foco inicial esta nas tags mais provaveis de piorar a qualidade da mascara.


In [ ]:
display(
    df_resumo_tag.loc[
        df_resumo_tag['tag_name'].isin(tags_prioritarias),
        ['nome_modelo', 'tag_name', 'tag_value', 'metric_name', 'mean', 'median', 'count'],
    ].sort_values(['tag_name', 'nome_modelo', 'metric_name', 'tag_value'])
)

for tag_name in tags_prioritarias:
    for metric_name in metric_names:
        fig, _ = plot_metric_tag_comparison(df_resumo_tag, metric_name, tag_name)
        plt.show()


## Distribuicao linha a linha

Os boxplots usam a base completa em memoria para mostrar dispersao e caudas, algo que os resumos persistidos nao capturam sozinhos.


In [ ]:
for metric_name in metric_names:
    fig, _ = plot_metric_distribution_by_model(df_base, metric_name)
    plt.show()

for tag_name in tags_prioritarias:
    fig, _ = plot_metric_distribution_by_tag(df_base, 'soft_dice', tag_name)
    plt.show()


## Leitura inicial

A combinacao entre medias por modelo, estabilidade por execucao, impacto das tags e distribuicao linha a linha ja permite responder quais modelos segmentam melhor e em que cenarios eles passam a falhar. A etapa seguinte continua sendo refinar os testes estatisticos no notebook 04 e aprofundar a leitura visual neste notebook 05.
